# BimBam Buy RAG Agent

Complete RAG pipeline with proper structure.

In [ ]:
import sys
!{sys.executable} -m pip install -q langchain-community pypdf fpdf2 cohere faiss-cpu langchain-google-genai langchain-text-splitters ipywidgets
print('✅ Dependencies installed')

In [ ]:
import os
from fpdf import FPDF

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

FILES = [
    (os.path.join(DATA_DIR, 'Guia de tiempos y costos.pdf'), 'envios'),
    (os.path.join(DATA_DIR, 'Manual de Garantia.pdf'), 'garantia'),
    (os.path.join(DATA_DIR, 'Politica de reembolsos.pdf'), 'reembolsos'),
    (os.path.join(DATA_DIR, 'Preguntas frecuentes.pdf'), 'faq'),
    (os.path.join(DATA_DIR, 'Programa de afiliados.pdf'), 'afiliados'),
]

def create_dummy_pdf(file_path, content):
    if not os.path.exists(file_path):
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font('Arial', size=12)
        pdf.multi_cell(0, 10, txt=content)
        pdf.output(file_path)
        print(f'Created: {file_path}')

for path, doc_type in FILES:
    content = f'Documentation for {doc_type}. Important policies and information about {doc_type}.'
    create_dummy_pdf(path, content)

print(f'✅ Files ready in: {DATA_DIR}')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_documents():
    all_docs = []
    for path, doc_type in FILES:
        try:
            loader = PyPDFLoader(path)
            docs = loader.load()
            for d in docs:
                d.metadata['source_doc'] = doc_type
                d.metadata['file_name'] = os.path.basename(path)
            all_docs.extend(docs)
            print(f'Loaded {len(docs)} pages from {doc_type}')
        except Exception as e:
            print(f'Error loading {path}: {e}')
    return all_docs

print('✅ load_documents() defined')

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=120,
        separators=['\n\n', '\n', '.', ' ']
    )
    chunks = splitter.split_documents(documents)
    print(f'Split into {len(chunks)} chunks')
    return chunks

print('✅ split_documents() defined')

In [ ]:
from langchain_community.embeddings import CohereEmbeddings

def create_embeddings(api_key):
    try:
        embeddings = CohereEmbeddings(
            cohere_api_key=api_key,
            model='embed-multilingual-v3.0',
            user_agent='langchain'
        )
        print('Embeddings initialized')
        return embeddings
    except Exception as e:
        print(f'Error: {e}')
        return None

print('✅ create_embeddings() defined')

In [ ]:
from langchain_community.vectorstores import FAISS

def create_vectorstore(chunks, embeddings):
    texts = [doc.page_content for doc in chunks]
    metadatas = [doc.metadata for doc in chunks]
    vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
    print(f'Vectorstore created with {len(chunks)} vectors')
    return vectorstore

print('✅ create_vectorstore() defined')

In [ ]:
def create_retriever(vectorstore):
    retriever = vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': 5}
    )
    print('Retriever created (k=5)')
    return retriever

print('✅ create_retriever() defined')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(retriever, google_api_key):
    llm = ChatGoogleGenerativeAI(
        model='gemini-1.5-pro',
        google_api_key=google_api_key,
        temperature=0.2
    )

    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are an expert on BimBam Buy policies. Use ONLY the provided context. Rules: Answer in Spanish. If insufficient info, say: No tengo informacion suficiente. Be clear and direct. Context: {context}'),
        ('human', '{question}')
    ])

    rag_chain = (
        RunnableParallel({
            'docs': lambda x: retriever.get_relevant_documents(x['question']),
            'question': RunnablePassthrough()
        })
        | RunnableParallel({
            'answer': (
                lambda x: {
                    'context': '\n\n'.join([d.page_content for d in x['docs']]),
                    'question': x['question']
                }
                | prompt
                | llm
                | StrOutputParser()
            ),
            'sources': lambda x: [{'doc': d.metadata['source_doc'], 'file': d.metadata['file_name']} for d in x['docs']]
        })
    )
    print('RAG chain created')
    return rag_chain

print('✅ create_rag_chain() defined')

In [ ]:
from google.colab import userdata

COHERE_API_KEY = userdata.get('COHERE_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

if not COHERE_API_KEY:
    raise ValueError('COHERE_API_KEY not found in Secrets')
if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY not found in Secrets')

print('✅ API keys loaded')

In [ ]:
print('Building RAG pipeline...\n')

docs = load_documents()
print()

chunks = split_documents(docs)
print()

embeddings = create_embeddings(COHERE_API_KEY)
print()

vectorstore = create_vectorstore(chunks, embeddings)
print()

retriever = create_retriever(vectorstore)
print()

rag_chain = create_rag_chain(retriever, GOOGLE_API_KEY)

print('\n' + '='*50)
print('✅ RAG Pipeline Ready!')
print('='*50)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_input = widgets.Text(
    placeholder='Ask about BimBam Buy policies...',
    description='Q:',
    layout=widgets.Layout(width='70%')
)

send_button = widgets.Button(
    description='Ask',
    button_style='info',
    icon='search'
)

output_area = widgets.Output()

def on_send_click(b):
    question = question_input.value.strip()
    
    if not question:
        with output_area:
            clear_output()
            print('⚠️ Please enter a question')
        return

    question_input.disabled = True
    send_button.disabled = True

    with output_area:
        clear_output()
        print('Processing...')

        try:
            result = rag_chain.invoke({'question': question})
            clear_output()
            print('RESPONSE:')
            print('-' * 60)
            print(result['answer'])
            print('-' * 60)
            print('\nSources:')
            sources_set = set()
            for src in result['sources']:
                sources_set.add(src['file'])
            for s in sorted(sources_set):
                print(f'  • {s}')
        except Exception as e:
            clear_output()
            print(f'Error: {str(e)}')

    question_input.disabled = False
    send_button.disabled = False
    question_input.value = ''

send_button.on_click(on_send_click)

input_box = widgets.HBox([question_input, send_button])
interface = widgets.VBox([input_box, output_area])

print('Chat ready! Ask questions below:')
display(interface)

In [ ]:
test_question = 'What are the main policies?'
print(f'Test: {test_question}\n')

result = rag_chain.invoke({'question': test_question})
print('Answer:')
print(result['answer'])